# TB2 – Machine Learning (1FIS0300)
## Modelo 2: Clustering Jerárquico Aglomerativo (Ward)
**Caso de Estudio:** Ministerio de Salud (MINSA) – Certificado de Nacido Vivo (CNV)  
**Objetivo:** Segmentación de perfiles maternos de riesgo para la prevención de bajo peso al nacer (< 2500 g) y prematuridad (< 37 semanas)  
**Responsable:** jfelix torres


### 1. Marco Metodológico y Justificación Arquitectónica

El **Clustering Jerárquico Aglomerativo** construye un árbol de agrupamiento (dendrograma) de abajo hacia arriba mediante fusión sucesiva de clusters según el criterio de mínima varianza interna (**Ward**).

#### Restricción de Escala y Solución de Muestreo:
* El cálculo de la matriz de distancias requiere $O(N^2)$ en memoria y $O(N^3)$ en tiempo.
* Para $N = 300,000$ observaciones, la matriz requeriría aproximadamente **180 GB de memoria RAM**, generando desbordamiento (`MemoryError`).
* **Estrategia adoptada:** 
  1. Se extrae una muestra maestra de **300 000 registros aleatorios** (coherente con la línea base del grupo).
  2. Se ajusta la matriz de enlace de Ward y se grafica el dendrograma sobre una **submuestra representativa de 30 000 registros**.
  3. Se propagan las etiquetas al universo de los 300 000 registros mediante `NearestCentroid` (clasificador por distancia mínima a centroides jerárquicos).
  4. Se reportan métricas tanto en la submuestra de entrenamiento como en la muestra global de 300 000.


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import silhouette_score, davies_bouldin_score

warnings.filterwarnings('ignore')

# Configuracion de reproducibilidad y visualizacion
SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']


### 2. Carga del Dataset y Resolución de Rutas
Búsqueda y carga del dataset limpio en formato Parquet (`NACIMIENTOS_CNV_LIMPIO.parquet`).


In [ ]:
RUTAS_DATASET = [
    '../NACIMIENTOS_CNV_LIMPIO.parquet',
    'NACIMIENTOS_CNV_LIMPIO.parquet',
    'dataset/NACIMIENTOS_CNV_LIMPIO.parquet',
    '/content/drive/MyDrive/NACIMIENTOS_CNV_LIMPIO.parquet'
]

ruta_encontrada = next((r for r in RUTAS_DATASET if os.path.exists(r)), None)
if ruta_encontrada is None:
    raise FileNotFoundError('No se localizó el archivo NACIMIENTOS_CNV_LIMPIO.parquet.')

print(f'Cargando dataset desde: {ruta_encontrada} ...')
t0 = time.time()
df_raw = pd.read_parquet(ruta_encontrada)
print(f'Dimensiones originales: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas ({time.time()-t0:.2f} s)')


### 3. Filtros Clínicos y Regla Anti-Fuga de Información

**Regla Anti-Fuga (Anti-Data Leakage):**
No se pueden utilizar como predictores las variables observadas en o después del nacimiento:
- `PESO_NACIDO`: Define el bajo peso.
- `TALLA_NACIDO`: Correlacionada directamente al nacimiento.
- `DUR_EMB_PARTO`: Define la prematuridad.

Estas variables solo se usan para extraer las etiquetas binarias de evaluación extrínseca:
- `y_bp`: Bajo Peso (< 2 500 g)
- `y_pre`: Prematuro (< 37 semanas)


In [ ]:
# Filtros de plausibilidad clínica
n_inicial = len(df_raw)
df_clean = df_raw.dropna(subset=['Bajo_Peso_Nacer', 'Prematuro'])
df_clean = df_clean[df_clean.Edad_Madre.between(10, 55)]
df_clean = df_clean[df_clean.PESO_NACIDO.between(300, 6500)]
df_clean = df_clean[df_clean.DUR_EMB_PARTO.between(22, 43)]

print(f'Registros limpios: {len(df_clean):,} (eliminados: {n_inicial - len(df_clean):,})')

# Extraccion de etiquetas clinicas para validacion extrinseca
y_bp_total = (df_clean.Bajo_Peso_Nacer == 'SI').astype('int8').values
y_pre_total = (df_clean.Prematuro == 'SI').astype('int8').values

print(f'Tasa global de Bajo Peso: {y_bp_total.mean():.2%}')
print(f'Tasa global de Prematuridad: {y_pre_total.mean():.2%}')


### 4. Muestreo Maestro de 300 000 Observaciones
Se toma una muestra aleatoria con `random_state=42` para garantizar compatibilidad con los modelos K-Means y DBSCAN.


In [ ]:
N_MUESTRA = 300_000
df_sample = df_clean.sample(n=N_MUESTRA, random_state=SEED)
y_bp_300k = (df_sample.Bajo_Peso_Nacer == 'SI').astype('int8').values
y_pre_300k = (df_sample.Prematuro == 'SI').astype('int8').values

print(f'Muestra maestra seleccionada: {len(df_sample):,} registros')
print(f'Tasa Bajo Peso (muestra): {y_bp_300k.mean():.2%}')
print(f'Tasa Prematuridad (muestra): {y_pre_300k.mean():.2%}')


### 5. Ingeniería de Características (`FEAT_CLU`)
Se transforman y codifican las 13 variables prenatales y maternas estandarizadas para el análisis no supervisado.


In [ ]:
EDU_MAP = {
    'NINGUN NIVEL/ILETRADO': 0, 'INCIAL/PRE-ESCOLAR': 0, 'PRIMARIA INCOMPLETA': 1,
    'PRIMARIA COMPLETA': 2, 'SECUNDARIA INCOMPLETA': 3, 'SECUNDARIA COMPLETA': 4,
    'SUPERIOR NO UNIV. INCOMPLETA': 5, 'SUPERIOR UNIV. INCOMPLETA': 5,
    'SUPERIOR NO UNIV. COMPLETA': 6, 'SUPERIOR UNIV. COMPLETA': 7
}
ORD5_MAP = {'1': 1, '2': 2, '3': 3, '4': 4, '>=5': 5}

def codificar_perdidas(val):
    if pd.isna(val) or val == 'NINGUNO': return 0
    if str(val).startswith('11'): return 11
    try: return int(val)
    except: return 0

REGION_MAP = {
    '01': 'SELVA', '02': 'SIERRA', '03': 'SIERRA', '04': 'SIERRA', '05': 'SIERRA', '06': 'SIERRA',
    '07': 'COSTA', '08': 'SIERRA', '09': 'SIERRA', '10': 'SIERRA', '11': 'COSTA', '12': 'SIERRA',
    '13': 'COSTA', '14': 'COSTA', '15': 'COSTA', '16': 'SELVA', '17': 'SELVA', '18': 'COSTA',
    '19': 'SIERRA', '20': 'COSTA', '21': 'SIERRA', '22': 'SELVA', '23': 'COSTA', '24': 'COSTA', '25': 'SELVA'
}
NO_CALIF = {'FAMILIAR', 'PARTERA', 'NADIE(AUTOAYUDA)', 'OTRO'}

X = pd.DataFrame(index=df_sample.index)
X['edad_madre']       = df_sample.Edad_Madre.astype('float32')
X['educacion']        = df_sample['Nivel_Intrucción_Madre'].astype(str).map(EDU_MAP).astype('float32')
X['n_embarazos']      = df_sample.Num_embar_madre.map(ORD5_MAP).astype('float32')
X['hijos_vivos']      = df_sample.Hijos_vivo_madre.map(ORD5_MAP).astype('float32')
X['perdidas_previas'] = df_sample.nacmuer_abort_madre.map(codificar_perdidas).clip(upper=3).astype('float32')
X['parto_multiple']   = (df_sample.Tipo_Parto.fillna('UNICO') != 'UNICO').astype('int8')
X['cesarea']          = (df_sample.Condicion_Parto == 'CESAREA').astype('int8')
X['fuera_eess']       = (df_sample.Lugar_Nacido != 'ESTABLECIMIENTO DE SALUD').astype('int8')
X['atencion_no_calif']= df_sample.Atiende_Parto.isin(NO_CALIF).astype('int8')
X['sis']              = (df_sample.Financiador_Parto == 'SIS').astype('int8')
X['extranjera']       = (df_sample.Pais_Madre != 'PERU').astype('int8')

depto = df_sample.IdUbigeoInei.astype(str).str[:2]
region = depto.map(REGION_MAP)
X['sierra']           = (region == 'SIERRA').astype('int8')
X['selva']            = (region == 'SELVA').astype('int8')

# Imputacion de mediana para valores faltantes
for col in ['educacion', 'n_embarazos', 'hijos_vivos', 'perdidas_previas']:
    X[col] = X[col].fillna(X[col].median())

FEAT_CLU = list(X.columns)
print(f'Variables de clustering configuradas ({len(FEAT_CLU)}):')
print(FEAT_CLU)


### 6. Estandarización y Submuestra de Entrenamiento Jerárquico
Se escalan los 300 000 registros con `StandardScaler`. Se extrae una submuestra representativa ($N = 30,000$) para el cálculo del enlace aglomerativo.


In [ ]:
scaler = StandardScaler()
X_scaled_300k = scaler.fit_transform(X).astype('float32')

# Submuestra representativa para el dendrograma y ajuste del arbol
N_SUB = 30_000
rng = np.random.default_rng(SEED)
idx_sub = rng.choice(N_MUESTRA, size=N_SUB, replace=False)

X_sub = X_scaled_300k[idx_sub]
y_bp_sub = y_bp_300k[idx_sub]
y_pre_sub = y_pre_300k[idx_sub]

print(f'Muestra completa estandarizada: {X_scaled_300k.shape}')
print(f'Submuestra de entrenamiento jerárquico: {X_sub.shape}')


### 7. Construcción del Dendrograma y Matriz de Enlace (Ward)
Calculamos la matriz de enlace aglomerativo con el criterio de mínima varianza de Ward y graficamos el dendrograma truncado.


In [ ]:
t0 = time.time()
print('Calculando matriz de enlace Ward sobre 30 000 observaciones...')
Z_ward = linkage(X_sub, method='ward', metric='euclidean')
t_linkage = time.time() - t0
print(f'Enlace Ward completado en {t_linkage:.2f} segundos.')

# Dendrograma truncado
plt.figure(figsize=(10, 5))
plt.title('Dendrograma de Clustering Jerárquico (Ward) - Submuestra 30k', fontsize=12, fontweight='bold')
dendrogram(
    Z_ward,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=45.,
    leaf_font_size=9.,
    show_contracted=True
)
plt.xlabel('Clusters condensados (número de observaciones)')
plt.ylabel('Distancia de disimilitud (Ward)')

# Linea de corte sugerida
corte_sugerido = 110.0
plt.axhline(y=corte_sugerido, color='crimson', linestyle='--', label=f'Corte sugerido (k=4, d={corte_sugerido})')
plt.legend()
plt.tight_layout()
plt.savefig('dendrograma_ward.png', dpi=150)
plt.show()


### 8. Evaluación de Soluciones para Distintos Valores de $k$
Analizamos el Coeficiente de Silueta y el Índice Davies-Bouldin para $k \in [3, 4, 5, 6]$.


In [ ]:
eval_k = []
# Evaluacion rapida de silueta sobre una muestra de evaluacion para agilidad
idx_eval = rng.choice(N_SUB, size=10_000, replace=False)
X_eval = X_sub[idx_eval]

for k_cand in [3, 4, 5, 6]:
    labs_cand = fcluster(Z_ward, t=k_cand, criterion='maxclust')
    sil = silhouette_score(X_eval, labs_cand[idx_eval])
    db = davies_bouldin_score(X_sub, labs_cand)
    eval_k.append({'k': k_cand, 'Silhouette': round(sil, 4), 'Davies_Bouldin': round(db, 4)})

df_eval_k = pd.DataFrame(eval_k)
print('Comparativa de particiones jerárquicas:')
print(df_eval_k.to_string(index=False))


### 9. Ajuste del Modelo Final y Propagación a los 300 000 Registros
Seleccionamos $k = 4$ clusters por equilibrio entre interpretabilidad clínica y separación matemática.
Propagamos las etiquetas a los 300 000 registros mediante `NearestCentroid`.


In [ ]:
K_OPT = 4
t0 = time.time()
agg_model = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward')
labels_sub = agg_model.fit_predict(X_sub)
t_fit = time.time() - t0

# Propagacion de centroides a los 300k
clf_centroides = NearestCentroid()
clf_centroides.fit(X_sub, labels_sub)

t0_prop = time.time()
labels_300k = clf_centroides.predict(X_scaled_300k)
t_prop = time.time() - t0_prop

print(f'Ajuste Ward en submuestra: {t_fit:.2f} s')
print(f'Propagación a 300 000 registros: {t_prop:.2f} s')
print('Distribución de observaciones en la muestra de 300k:')
print(pd.Series(labels_300k).value_counts().sort_index())


### 10. Métricas Intrínsecas del Modelo Jerárquico
Cálculo de métricas cuantitativas sobre la submuestra y la muestra global.


In [ ]:
# Evaluacion de silueta en 15k observaciones para comparacion estandar
idx_sil = rng.choice(N_MUESTRA, size=15_000, replace=False)
sil_global = silhouette_score(X_scaled_300k[idx_sil], labels_300k[idx_sil])
db_global = davies_bouldin_score(X_scaled_300k, labels_300k)

print('=== RESULTADOS CUANTITATIVOS (CLUSTERING JERÁRQUICO - WARD) ===')
print(f'Número de clusters (k): {K_OPT}')
print(f'Coeficiente de Silueta: {sil_global:.4f}')
print(f'Índice Davies-Bouldin:  {db_global:.4f}')
print(f'Tiempo total cómputo:   {t_fit + t_prop:.2f} s')


### 11. Validación Clínica Extrínseca: Perfiles de Riesgo Neonatal
Evaluamos la concentración de **Bajo Peso al Nacer** y **Prematuridad** en cada uno de los 4 clusters identificados.


In [ ]:
df_res = pd.DataFrame({
    'cluster': labels_300k,
    'bajo_peso': y_bp_300k,
    'prematuro': y_pre_300k
})

perfil_riesgo = df_res.groupby('cluster').agg(
    total=('bajo_peso', 'count'),
    tasa_bajo_peso=('bajo_peso', 'mean'),
    tasa_prematuro=('prematuro', 'mean')
).reset_index()

perfil_riesgo['pct_poblacion'] = perfil_riesgo['total'] / len(df_res) * 100
perfil_riesgo['tasa_bajo_peso_pct'] = perfil_riesgo['tasa_bajo_peso'] * 100
perfil_riesgo['tasa_prematuro_pct'] = perfil_riesgo['tasa_prematuro'] * 100

print('=== TASAS DE EVENTOS CLÍNICOS POR CLUSTER ===')
print(perfil_riesgo[['cluster', 'total', 'pct_poblacion', 'tasa_bajo_peso_pct', 'tasa_prematuro_pct']].round(2).to_string(index=False))

# Visualizacion de tasas
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=perfil_riesgo, x='cluster', y='tasa_bajo_peso_pct', ax=ax[0], palette=PALETTE)
ax[0].set_title('Tasa de Bajo Peso (< 2500 g) por Cluster', fontweight='bold')
ax[0].set_ylabel('% Bajo Peso')
ax[0].axhline(y_bp_300k.mean() * 100, color='red', linestyle='--', label=f'Promedio nacional ({y_bp_300k.mean():.1%})')
ax[0].legend()

sns.barplot(data=perfil_riesgo, x='cluster', y='tasa_prematuro_pct', ax=ax[1], palette=PALETTE)
ax[1].set_title('Tasa de Prematuridad (< 37 s) por Cluster', fontweight='bold')
ax[1].set_ylabel('% Prematuro')
ax[1].axhline(y_pre_300k.mean() * 100, color='red', linestyle='--', label=f'Promedio nacional ({y_pre_300k.mean():.1%})')
ax[1].legend()

plt.tight_layout()
plt.savefig('tasas_riesgo_clusters.png', dpi=150)
plt.show()


### 12. Caracterización Multivariable de los Perfiles Clínicos
Analizamos los promedios de las variables originales para interpretar clínicamente cada grupo.


In [ ]:
df_analisis = X.copy()
df_analisis['cluster'] = labels_300k

medias_cluster = df_analisis.groupby('cluster').mean()
print('=== PROMEDIO DE VARIABLES ORIGINALES POR CLUSTER ===')
print(medias_cluster.T.round(2))

# Heatmap de perfiles estandarizados por cluster
medias_z = pd.DataFrame(X_scaled_300k, columns=FEAT_CLU).groupby(labels_300k).mean()
plt.figure(figsize=(12, 4))
sns.heatmap(medias_z, cmap='coolwarm', annot=True, fmt='.2f', cbar=True, center=0)
plt.title('Desviaciones Estándar de Características por Cluster (Perfiles Z-Score)', fontsize=12, fontweight='bold')
plt.xlabel('Variables de Clustering')
plt.ylabel('Cluster Jerárquico')
plt.tight_layout()
plt.savefig('heatmap_perfiles_z.png', dpi=150)
plt.show()


### 13. Conclusiones y Recomendaciones para la Gestión en el MINSA

1. **Eficiencia y Viabilidad Técnica:** Se demostró que mediante una submuestra calibrada de 30 000 datos junto a propagación de centroides se supera la limitación cuadrática $O(N^2)$ del clustering jerárquico, permitiendo clasificar 300 000 historias clínicas en segundos sin error de desbordamiento de memoria.
2. **Diferenciación Clínica de Riesgo:** El algoritmo jerárquico separó con nitidez subgrupos de gestantes con factores de riesgo acumulados (embarazo múltiple, antecedentes de pérdidas, dispersión geográfica), concentrando tasas de prematurez y bajo peso sensiblemente superiores al promedio nacional.
3. **Aplicación para Políticas Públicas:** El MINSA puede utilizar estas reglas de asignación jerárquica en los puestos de salud de primer nivel para priorizar controles prenatales intensivos y planificar camas de cuidados intensivos neonatales (UCIN).
